In [1]:
from src.basemodel import Capa, Clasificacion,Familia,Articulo,Segmento,Clase
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
import psycopg2
from src.database import create_connection
from src.ai import generate_prompt,generar_familias_prompt, generar_clases_prompt
import json

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_IA_MODEL = os.getenv("GEMINI_IA_MODEL")

client = genai.Client(api_key=GEMINI_API_KEY)

segmentos:list[Segmento] = []
articulos: list[Articulo] = []
familias: list[Familia] = []
clases:list[Clase] = []

capa1_segmentos:dict[str,Capa[Segmento]] = {}
capa2_familias:dict[str,Capa[Familia]] = {}
capa3_clases:dict[str,Capa[Clase]] = {}
capa4_articulos:dict[str,list[Articulo]] = {}

catalogo: dict[str,list[Familia]] = []

articulos = [
    Articulo("adajk","coca cola 1/2"),
    Articulo("calmkllwa","gaseosa kr 300 ml"),
    Articulo("aadac","cuarto de pollo"),
    Articulo("alwekmav","delivery domicilio"),
    Articulo("acmwakoi","chaufa + pollo"),
    Articulo("aaasdc","tallarin saltado")
]


In [2]:
#traer segmentos
conn = create_connection()
with conn.cursor() as cur:
    cur.execute("select id,descripcion from segmento")
    result = cur.fetchall()
conn.close()
segmentos = [Familia(id,nombre) for id,nombre in result]

In [3]:
##generar segmentos
prompt = generate_prompt(
        (seg.nombre for seg in segmentos),
        (art.nombre for art in articulos)
    )

response = client.models.generate_content(
    contents=prompt,
    model=GEMINI_IA_MODEL,
    config={
        "response_mime_type":"application/json",
        "response_schema":list[Clasificacion]
    }
)
data: list[Clasificacion] = response.parsed

In [4]:
dict_articulos = {}
for i,articulo in enumerate(articulos):
    dict_articulos[i] = articulo

In [5]:
##crear capa1 segmentos
for match in data:
    indexSegmento:int = match.id_grupo
    segmento:Familia = segmentos[indexSegmento]
    articulo:Articulo = articulos[match.id_articulo]
    capa1_segmentos.setdefault(segmento.id,Capa[Familia](segmento,indexSegmento))
    capa1_segmentos.get(segmento.id).items.append(articulo)

In [6]:
#obtener familias
segmento_ids = tuple(id[:2] for id in capa1_segmentos.keys())
query = """ select id,descripcion from familia where {} """.format(" OR ".join(["id like %s"] * len(segmento_ids)))
params = tuple(f"{id}%" for id in segmento_ids)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
familias = [Familia(id,nombre) for id,nombre in result]

In [7]:
#agregar familias a segmentos
for capa in capa1_segmentos.values(): capa.value.familias = []
for fam in familias:
    tramoSegmento = fam.id[:2]
    segmento = next((s.value for s in capa1_segmentos.values() if s.value.id.startswith(tramoSegmento)),None)
    if segmento is not None: segmento.familias.append(fam)

In [8]:
#generar familias
data: dict[str,list[Clasificacion]] = {}
for seg in capa1_segmentos.values():
    prompt = generar_familias_prompt(
        [fam.nombre for fam in seg.value.familias],
        [art.nombre for art in seg.items],
    )
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    data[seg.value.id] = response.parsed

In [9]:
#crear capa2_familias
for idSegmento,res in data.items():
    capaSegmento: Capa[Segmento] = capa1_segmentos.get(idSegmento,None)
    if capaSegmento is None: continue
    for clasificacion in res:
        familia = capaSegmento.value.familias[clasificacion.id_grupo]
        articulo = capaSegmento.items[clasificacion.id_articulo]
        if familia is None or articulo is None: continue
        if capaSegmento.subcapas.get(familia.id) is None:
            capa = Capa[Familia](familia,clasificacion.id_grupo)
            capaSegmento.subcapas[familia.id] = capa
            capa2_familias[familia.id] = capa
        capaFamilia: Capa[Familia] = capaSegmento.subcapas.get(familia.id)
        capaFamilia.items.append(articulo)

In [10]:
#obtener clases
familia_ids = tuple(id[:4] for id in capa2_familias.keys())
query = """ select id,descripcion from clase where {} """.format(" OR ".join(["id like %s"] * len(familia_ids)))
params = tuple(f"{id}%" for id in familia_ids)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
clases = [Clase(id,nombre) for id,nombre in result]

In [11]:
#agregar clases a familias
for capa in capa2_familias.values(): capa.value.clases = []
for cla in clases:
    tramoFamilia = cla.id[:4]
    familia = next((s.value for s in capa2_familias.values() if s.value.id.startswith(tramoFamilia)),None)
    if familia is not None: familia.clases.append(cla)

In [ ]:
#generar clases
data: dict[str,list[Clasificacion]] = {}
for seg in capa2_familias.values():
    prompt = generar_clases_prompt(
        [cla.nombre for cla in seg.value.clases],
        [art.nombre for art in seg.items],
    )
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    data[seg.value.id] = response.parsed

---------------------------

Eres un clasificador del catálogo SUNAT.

Debes clasificar cada artículo en UNA de las clases. los articulos pertencen a una empresa la cual es un restaurante

clases:

0: Café y té
1: Bebidas alcohólicas
2: Bebidas no alcohólicas
3: Jugos o concentrados de cítricos frescos
4: Jugos o concentrados de bayas frescas
5: Jugos o concentrados de especies con pepitas frescas
6: Jugos o concentrados de especies con una semilla grande frescas
7: Jugos o concentrados de especies tropicales frescas
8: Jugos o concentrados de melones frescos

Artículos:

id_articulo: 0, nombre:coca cola 1/2
id_articulo: 1, nombre:gaseosa kr 300 ml

Responde únicamente JSON.

Formato:

[
    {
        "id_articulo": 1,
        "id_grupo": 14,
        "confianza": 0.98
    }
]
    
---------------------------

Eres un clasificador del catálogo SUNAT.

Debes clasificar cada artículo en UNA de las clases. los articulos pertencen a una empresa la cual es un restaurante

clases:

0: Carne y

In [13]:
#crear capa3 clases
for idFamilia,res in data.items():
    capaFamilia: Capa[Familia] = capa2_familias.get(idFamilia,None)
    if capaFamilia is None: continue
    for clasificacion in res:
        clase = capaFamilia.value.clases[clasificacion.id_grupo]
        articulo = capaFamilia.items[clasificacion.id_articulo]
        if clase is None and articulo is None: continue
        if capaFamilia.subcapas.get(clase.id) is None:
            capa = Capa[Clase](clase,clasificacion.id_grupo)
            capaFamilia.subcapas[clase.id] = capa
            capa3_clases[clase.id] = capa
        capaClase: Capa[Clase] = capaFamilia.subcapas.get(clase.id)
        capaClase.items.append(articulo)

In [14]:
for capa in capa3_clases.values():
    for articulo in capa.items:
        print( f"{capa.value.id}:{capa.value.nombre}, {articulo.id}:{articulo.nombre}")

50202300:Bebidas no alcohólicas, adajk:coca cola 1/2
50202300:Bebidas no alcohólicas, calmkllwa:gaseosa kr 300 ml
50111500:Carne y aves de corral, aadac:cuarto de pollo
50192700:Platos combinados empaquetados, acmwakoi:chaufa + pollo
50192700:Platos combinados empaquetados, aaasdc:tallarin saltado
78101800:Transporte de carga por carretera, alwekmav:delivery domicilio


In [23]:
data

{'50200000': [Clasificacion(id_articulo=0, id_grupo=2, confianza=0.99),
  Clasificacion(id_articulo=1, id_grupo=2, confianza=0.99)],
 '50110000': [Clasificacion(id_articulo=0, id_grupo=0, confianza=0.99)],
 '50190000': [Clasificacion(id_articulo=0, id_grupo=6, confianza=0.95),
  Clasificacion(id_articulo=1, id_grupo=6, confianza=0.92)],
 '78100000': [Clasificacion(id_articulo=0, id_grupo=7, confianza=0.95)]}